# Agentic RAG with LangGraph — The Modern Agent-Loop Pattern

**Stack:** LangGraph + LangChain + Qdrant + Gemini + Tavily

Based on the [Qdrant Agentic RAG with LangGraph tutorial](https://qdrant.tech/documentation/tutorials-build-essentials/agentic-rag-langgraph/), adapted for Gemini.

## What you will learn

1. What **Agentic RAG** really means in 2026 — the LLM + tool loop, not hand-coded grader nodes
2. How to store knowledge in **Qdrant** (vector database)
3. How to give an agent **multiple retrieval tools** and let it pick the right one
4. How to add a **web search** tool for questions outside the knowledge base
5. How to build the whole thing as a tiny LangGraph with just two nodes: `agent` and `tools`

## 1. What is Agentic RAG (the modern take)

Plain RAG always retrieves, every time, from one source. That is fine but dumb.

**Agentic RAG** means the LLM is the agent and it decides, at every turn:
- Do I even need to retrieve, or can I answer directly?
- If I retrieve, **which source** should I use? (docs A? docs B? the web?)
- Are these docs enough, or should I call another tool with a better query?
- When do I stop and answer?

That is it. No separate grader node. No separate rewriter node. Modern LLMs (Gemini 3, Claude 4.x, GPT-5.x) do all of that *inside* one tool-calling loop if you:
1. Give them the right tools
2. Write a clear system prompt
3. Let them loop until they are done

The graph looks like this — only two nodes:

```
START -> agent -> (tool_calls?) -> tools -> agent -> ... -> END
```

The agent keeps looping back to itself through `tools` until it has enough info to answer.

## 2. Install Libraries

| Library | Purpose |
|---|---|
| `langgraph` | Build the agent graph |
| `langchain` | Core primitives (messages, tools) |
| `langchain-google-genai` | Gemini LLM + embeddings |
| `langchain-qdrant` | Qdrant integration for LangChain |
| `qdrant-client` | Qdrant Python client |
| `langchain-text-splitters` | Chunk long documents |
| `tavily-python` | Web search tool |
| `python-dotenv` | Load `.env` file |

Run this cell **once**:

In [ ]:
# !uv add langgraph langchain langchain-google-genai langchain-qdrant qdrant-client langchain-text-splitters tavily-python python-dotenv
# or with pip:
# !pip install -qU langgraph langchain langchain-google-genai langchain-qdrant qdrant-client langchain-text-splitters tavily-python python-dotenv

## 3. Load API Keys

Create `.env` in the project root:

```
GEMINI_API_KEY=your_gemini_key
TAVILY_API_KEY=your_tavily_key   # optional, only for the web search tool
```

- Gemini key: [https://aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)
- Tavily key: [https://tavily.com](https://tavily.com)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

assert GEMINI_API_KEY, "GEMINI_API_KEY is missing. Add it to your .env file."

# langchain-google-genai reads from GOOGLE_API_KEY
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

print("Gemini key loaded.")
print("Tavily key loaded." if TAVILY_API_KEY else "Tavily key missing - web search tool will be skipped.")

## 4. Setup the LLM and the Embedding Model

- **LLM (the agent brain):** `gemini-3-flash-preview` — fast and great at tool calling.
- **Embeddings:** `gemini-embedding-2-preview` at 768 dimensions — smaller and faster than the full 3072.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0,
)

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    output_dimensionality=768,
)

print("LLM and embeddings ready.")

## 5. Build Two Knowledge Bases

The key idea in this tutorial: we give the agent **two different retrieval tools** so it has to *choose*. One tool covers **LangGraph / LangChain**, the other covers **Qdrant / vector databases**. The agent picks based on the question.

In a real project, each knowledge base could be a separate collection of PDFs, Notion pages, etc.

In [ ]:
from langchain_core.documents import Document

langgraph_knowledge = [
    "LangGraph is a low-level orchestration framework built on LangChain. It lets you design AI agents as a graph of nodes and edges, where each node is a Python function. LangGraph supports cycles, so an agent can loop until a condition is met.",
    "In LangGraph, a StateGraph holds a shared State object that every node reads and writes to. The most common state is MessagesState, which carries a list of chat messages between nodes.",
    "A tool in LangChain is any Python function decorated with @tool. When bound to an LLM via bind_tools, the model can emit a tool_call and LangGraph's ToolNode will execute the function and return a ToolMessage.",
    "Conditional edges in LangGraph let the graph choose the next node based on the current state. A typical agent pattern routes from the agent node to a tools node when the last LLM message contains tool_calls, and to END otherwise.",
    "LangGraph exposes StateGraph, START, and END from langgraph.graph. Prebuilt helpers like ToolNode and create_react_agent live in langgraph.prebuilt and save a lot of boilerplate.",
    "To run a LangGraph agent, compile the graph with graph = workflow.compile() and then call graph.invoke({'messages': [...]}) or stream updates with graph.stream(...).",
]

qdrant_knowledge = [
    "Qdrant is an open-source vector database written in Rust. It stores embeddings along with metadata and supports fast approximate nearest-neighbor search using the HNSW algorithm.",
    "Qdrant organizes vectors into collections. Each collection has a fixed vector size and a distance metric (Cosine, Dot, or Euclidean). You must match the vector size to your embedding model's output dimension.",
    "You can run Qdrant three ways: in-memory via QdrantClient(':memory:') for tests, in a local Docker container on port 6333, or as a managed service on Qdrant Cloud. The LangChain API is identical in all three cases.",
    "The langchain-qdrant package provides QdrantVectorStore, which wraps a QdrantClient so it behaves like any LangChain vector store. You can call add_documents, similarity_search, or as_retriever on it.",
    "Qdrant supports metadata filtering, hybrid (dense plus sparse) search, and payload indexing. This makes it possible to mix semantic search with exact filters like user_id == 42 or language == 'en'.",
    "Cosine distance is the most common metric for text embeddings because it measures the angle between vectors and ignores magnitude. Use Distance.COSINE in VectorParams when creating a Qdrant collection for most NLP use cases.",
]

langgraph_docs = [Document(page_content=t, metadata={"source": "langgraph"}) for t in langgraph_knowledge]
qdrant_docs = [Document(page_content=t, metadata={"source": "qdrant"}) for t in qdrant_knowledge]

print(f"LangGraph KB: {len(langgraph_docs)} docs")
print(f"Qdrant KB:    {len(qdrant_docs)} docs")

## 6. Chunk the Documents

Our docs are small so this barely splits anything, but this is exactly how you would handle real PDFs.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

langgraph_chunks = splitter.split_documents(langgraph_docs)
qdrant_chunks = splitter.split_documents(qdrant_docs)

print(f"LangGraph chunks: {len(langgraph_chunks)}")
print(f"Qdrant chunks:    {len(qdrant_chunks)}")

## 7. Create Two Qdrant Collections

One in-memory Qdrant client, two collections (one per knowledge base). For production, swap `":memory:"` for a URL:

```python
client = QdrantClient(url="http://localhost:6333")            # local Docker
client = QdrantClient(url="...", api_key="...")              # Qdrant Cloud
```

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(":memory:")

def make_collection(name: str, chunks: list[Document]) -> QdrantVectorStore:
    client.create_collection(
        collection_name=name,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    )
    store = QdrantVectorStore(client=client, collection_name=name, embedding=embeddings)
    store.add_documents(chunks)
    return store

langgraph_store = make_collection("langgraph_kb", langgraph_chunks)
qdrant_store = make_collection("qdrant_kb", qdrant_chunks)

print("Both collections created and populated.")

## 8. Turn Each Vector Store into a Retriever Tool

`create_retriever_tool` is a LangChain helper that wraps a retriever into a tool the LLM can call. The **description** is critical — the LLM reads it to decide *when* to use this tool, so make the descriptions clear and non-overlapping.

In [ ]:
from langchain_core.tools.retriever import create_retriever_tool

langgraph_retriever = langgraph_store.as_retriever(search_kwargs={"k": 3})
qdrant_retriever = qdrant_store.as_retriever(search_kwargs={"k": 3})

langgraph_tool = create_retriever_tool(
    langgraph_retriever,
    name="search_langgraph_docs",
    description=(
        "Search documentation about LangGraph and LangChain: nodes, edges, state, "
        "tools, messages, agents, and how to build graphs. Use this for any question "
        "about LangGraph or LangChain concepts."
    ),
)

qdrant_tool = create_retriever_tool(
    qdrant_retriever,
    name="search_qdrant_docs",
    description=(
        "Search documentation about Qdrant and vector databases: collections, "
        "embeddings, distance metrics, HNSW, filtering, and hybrid search. Use this "
        "for any question about Qdrant or how vector databases work."
    ),
)

## 9. Web Search Tool (Tavily)

For questions that are not in either knowledge base (latest news, general facts), the agent falls back to a web search.

If you didn't set `TAVILY_API_KEY`, this cell skips the web tool — the agent will still work with just the two retrievers.

In [ ]:
from langchain.tools import tool

web_search_tool = None

if TAVILY_API_KEY:
    from tavily import TavilyClient
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

    @tool
    def web_search(query: str) -> str:
        """Search the web for recent information or topics outside the LangGraph 
        and Qdrant knowledge bases. Use this for current events, news, or any 
        general question that the other tools cannot answer.

        Args:
            query: the search query.
        """
        result = tavily_client.search(query=query, search_depth="basic", max_results=3)
        hits = result.get("results", [])
        return "\n\n".join(f"- {h['title']}\n  {h['content']}" for h in hits) or "No results."

    web_search_tool = web_search
    print("Web search tool ready.")
else:
    print("Skipping web search tool (no TAVILY_API_KEY).")

## 10. Bind All Tools to the LLM

This is where Agentic RAG comes alive — the LLM now *knows* about three (or two) tools and can decide which to call.

In [ ]:
tools = [langgraph_tool, qdrant_tool]
if web_search_tool is not None:
    tools.append(web_search_tool)

llm_with_tools = llm.bind_tools(tools)

print("Tools bound to LLM:")
for t in tools:
    print(f"  - {t.name}")

## 11. Define the Graph State

Our state is just a list of messages that grows as nodes talk to each other. `add_messages` is a reducer — it **appends** new messages to the list instead of replacing it.

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

## 12. The Agent Node

One function. The LLM looks at the conversation so far and either:
- Emits a final answer, or
- Emits one or more `tool_calls` asking to use a tool.

A good **system prompt** is what lets us skip the grader/rewriter nodes — it tells the LLM how to behave across the loop.

In [ ]:
from langchain.messages import SystemMessage

SYSTEM_PROMPT = """You are a helpful research assistant with access to three tools:
- search_langgraph_docs: for LangGraph / LangChain questions
- search_qdrant_docs: for Qdrant / vector database questions
- web_search: for anything else, current events, or when the other tools return nothing useful

Rules:
1. If a question is small talk or a greeting, answer directly without any tool.
2. Pick the MOST specific tool for the question. Use web_search only as a fallback.
3. If the first tool call does not give you a clear answer, try a different query or a different tool.
4. Cite which tool(s) you used at the end of your answer.
5. If you still cannot answer after trying, say so honestly.
"""

def agent(state: State):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

## 13. The Tool Node (built from scratch to show the mechanics)

Whenever the LLM emits `tool_calls`, this node runs each one and returns a `ToolMessage` with the result. LangGraph ships a prebuilt `ToolNode` that does exactly this — we will use the handwritten version to teach, then mention the prebuilt alternative.

In [ ]:
import json
from langchain.messages import ToolMessage

class BasicToolNode:
    """Runs tool calls found on the last AI message."""

    def __init__(self, tools: list):
        self.tools_by_name = {t.name: t for t in tools}

    def __call__(self, state: State):
        last_msg = state["messages"][-1]
        outputs = []
        for tool_call in last_msg.tool_calls:
            tool_fn = self.tools_by_name[tool_call["name"]]
            result = tool_fn.invoke(tool_call["args"])
            outputs.append(
                ToolMessage(
                    content=json.dumps(result) if not isinstance(result, str) else result,
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )
        return {"messages": outputs}

tool_node = BasicToolNode(tools)

# Production alternative (one line):
# from langgraph.prebuilt import ToolNode
# tool_node = ToolNode(tools)

## 14. The Routing Function

After every `agent` step we ask: did the LLM emit any tool calls?
- **Yes** -> go to `tools` node, execute them, loop back to `agent`.
- **No** -> the LLM gave a final answer, so we stop.

In [ ]:
from langgraph.graph import END

def route(state: State):
    last_msg = state["messages"][-1]
    if getattr(last_msg, "tool_calls", None):
        return "tools"
    return END

## 15. Build the Graph

Just two nodes and one cycle:

```
START -> agent -> (tool_calls?) -> tools -> agent -> ... -> END
```

In [ ]:
from langgraph.graph import StateGraph, START

graph_builder = StateGraph(State)

graph_builder.add_node("agent", agent)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "agent")
graph_builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
graph_builder.add_edge("tools", "agent")

graph = graph_builder.compile()
print("Graph compiled.")

## 16. Visualize

See the two-node loop for yourself.

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

## 17. Run the Agent

We stream updates so you can see **every tool the agent picks and in what order**. This is the moment where Agentic RAG becomes real.

In [ ]:
def run_agent(question: str):
    print(f"USER: {question}\n{'-' * 60}")
    for event in graph.stream({"messages": [("user", question)]}):
        for node_name, update in event.items():
            print(f"[node: {node_name}]")
            update["messages"][-1].pretty_print()
            print()

In [ ]:
# Test 1: LangGraph question -> agent should pick search_langgraph_docs
run_agent("What is a conditional edge in LangGraph and when would I use one?")

In [ ]:
# Test 2: Qdrant question -> agent should pick search_qdrant_docs
run_agent("Which distance metric should I use in Qdrant for text embeddings, and why?")

In [ ]:
# Test 3: Multi-tool question -> agent may call BOTH retrievers
run_agent("How do I wire a Qdrant retriever into a LangGraph agent as a tool?")

In [ ]:
# Test 4: Outside the knowledge base -> agent should use web_search (if Tavily key is set)
run_agent("What is the current weather in Karachi today?")

In [ ]:
# Test 5: Small talk -> agent should NOT call any tool
run_agent("Hi! How are you today?")

## 18. Bonus — The One-Liner Version

Everything we just built (agent node + tool node + route + graph) is so common that LangGraph provides `create_react_agent`. It produces the **same** graph with one line:

```python
from langgraph.prebuilt import create_react_agent

quick_agent = create_react_agent(llm, tools=tools, prompt=SYSTEM_PROMPT)
quick_agent.invoke({"messages": [("user", "What is a StateGraph?")]})
```

You now know what is inside it.

In [ ]:
from langgraph.prebuilt import create_react_agent

quick_agent = create_react_agent(llm, tools=tools, prompt=SYSTEM_PROMPT)

result = quick_agent.invoke({"messages": [("user", "What is a StateGraph in LangGraph?")]})
result["messages"][-1].pretty_print()

## 19. Recap

- **Agentic RAG = LLM with tools in a loop.** No graders, no rewriters — the LLM handles all of that inside its own reasoning.
- We built **two Qdrant collections**, wrapped each as a retriever tool, and added a web search fallback.
- The graph has just **two nodes** (`agent`, `tools`) and **one cycle**.
- A strong **system prompt** is what makes the agent reliable — it is the 'brain' of the loop.
- `create_react_agent` is the production one-liner, but now you know exactly what it hides.

### Exercises for students

1. **Add a third knowledge base.** Create a Qdrant collection about Pakistani tech companies (Careem, Bykea, Bano Qabil) and add it as a fourth tool.
2. **Persistent Qdrant.** Run `docker run -p 6333:6333 qdrant/qdrant` and change the client to `QdrantClient(url="http://localhost:6333")`.
3. **Add a calculator tool.** The agent should pick math tool, retrievers, or web search depending on the question.
4. **Limit the loop.** Count agent steps in the state and cut off after 5 iterations.
5. **Swap to `gemini-3-pro-preview`** and compare how well it picks tools vs the flash model.

### Further reading

- [Qdrant: Agentic RAG with LangGraph](https://qdrant.tech/documentation/tutorials-build-essentials/agentic-rag-langgraph/)
- [LangGraph docs](https://docs.langchain.com/oss/python/langgraph/)
- [Qdrant LangChain integration](https://docs.langchain.com/oss/python/integrations/vectorstores/qdrant)
- [GoogleGenerativeAIEmbeddings](https://docs.langchain.com/oss/python/integrations/embeddings/google_generative_ai)